# AutoAttack Low-ε Sweep — VGG-16 on BelgiumTSC

Fine-grained sweep over sub-pixel ε values to find the robustness floor.
ε range: 0.0005 – 0.01 (normalized space), ~0.03 – 0.58 pixels out of 255.
Same `NormalizedModel` wrapper and eval subset (seed=42, N=2000) as the standard AutoAttack notebook.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
from torchvision.models import VGG16_Weights
from torch.utils.data import DataLoader, Dataset, Subset
import pandas as pd
import numpy as np
from PIL import Image
import os
import io
import re
import json
import contextlib
import matplotlib.pyplot as plt

DATA_DIR    = 'dataset'
TEST_DIR    = os.path.join(DATA_DIR, 'BelgiumTSC_Testing', 'Testing')
MODEL_NAME  = 'VGG-16'
DATASET     = 'BelgiumTSC'
CKPT_PATH   = 'best_vgg16_btsd.pth'
NUM_CLASSES = 62
BATCH_SIZE  = 128
SEED        = 42
EVAL_SUBSET = 2000

EPS_NORMALIZED = [0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.0075, 0.01]
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')
print(f'Eval subset: {EVAL_SUBSET} | EPS (normalized): {EPS_NORMALIZED}')
print(f'eps_pixel range: {EPS_NORMALIZED[0]*np.mean(IMAGENET_STD):.6f} – {EPS_NORMALIZED[-1]*np.mean(IMAGENET_STD):.5f}')


In [ ]:
class NumericImageFolder(torchvision.datasets.ImageFolder):
    def find_classes(self, directory):
        classes = sorted(
            (e.name for e in os.scandir(directory) if e.is_dir()),
            key=lambda x: int(x)
        )
        class_to_idx = {cls: int(cls) for cls in classes}
        return classes, class_to_idx

raw_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
full_test_dataset = NumericImageFolder(TEST_DIR, transform=raw_transform, allow_empty=True)

torch.manual_seed(SEED)
indices = torch.randperm(len(full_test_dataset))[:EVAL_SUBSET].tolist()
eval_dataset = Subset(full_test_dataset, indices)
eval_loader  = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images_list, labels_list = [], []
for imgs, lbls in eval_loader:
    images_list.append(imgs)
    labels_list.append(lbls)
images = torch.cat(images_list).to(device)
labels = torch.cat(labels_list).to(device)

print(f'Eval samples: {len(images)} | Image shape: {images.shape}')
print(f'Pixel range:  [{images.min():.3f}, {images.max():.3f}]  (expected [0, 1])')

# Load base model
model = models.vgg16(weights=None)
model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model = model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)


class NormalizedModel(nn.Module):
    """Wraps a model trained on ImageNet-normalized inputs.
    AutoAttack feeds images in [0, 1] pixel space; this wrapper normalizes internally."""
    def __init__(self, base_model, mean, std):
        super().__init__()
        self.model = base_model
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return self.model((x - self.mean) / self.std)


wrapped_model = NormalizedModel(model, IMAGENET_MEAN, IMAGENET_STD).to(device)
wrapped_model.eval()
for p in wrapped_model.parameters():
    p.requires_grad_(False)

correct = 0
with torch.no_grad():
    for i in range(0, len(images), BATCH_SIZE):
        preds = wrapped_model(images[i:i+BATCH_SIZE]).argmax(1)
        correct += (preds == labels[i:i+BATCH_SIZE]).sum().item()
clean_acc = correct / len(images)
print(f'Clean accuracy (wrapped model): {clean_acc:.4f} ({clean_acc*100:.2f}%)')


In [ ]:
normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
test_batch = images[:16]

with torch.no_grad():
    preds_wrapped = wrapped_model(test_batch).argmax(1)

normalized_batch = torch.stack([normalize(img.cpu()) for img in test_batch]).to(device)
with torch.no_grad():
    preds_manual = model(normalized_batch).argmax(1)

assert (preds_wrapped == preds_manual).all(), \
    'FAIL: wrapper predictions differ from the manually-normalized pipeline!'
print('PASS: NormalizedModel wrapper is correct.')
print(f'Sample predictions: {preds_wrapped[:8].tolist()}')


In [ ]:
# Part 1 — AutoAttack low-ε sweep
# verbose=True so sub-attack cascade is visible; we also capture stdout for Part 4 parsing
from autoattack import AutoAttack

sweep_results = {}   # eps_norm -> {robust_acc, asr, log}
print(f'Running AutoAttack L-inf sweep over {EPS_NORMALIZED}')
print('Very small eps may still achieve 100% ASR (sub-pixel perturbations are sufficient).')
print('=' * 72)

for eps_norm in EPS_NORMALIZED:
    eps_pixel = float(eps_norm * np.mean(IMAGENET_STD))
    px_of_255 = eps_pixel * 255
    print(f'\n--- eps_norm={eps_norm}  eps_pixel={eps_pixel:.6f}  ({px_of_255:.4f}/255) ---')

    # Capture verbose log for sub-attack parsing
    log_buf = io.StringIO()
    with contextlib.redirect_stdout(log_buf):
        adversary = AutoAttack(
            wrapped_model, norm='Linf', eps=eps_pixel,
            version='standard', verbose=True,
        )
        x_adv = adversary.run_standard_evaluation(images, labels, bs=64)
    log_text = log_buf.getvalue()
    print(log_text)  # re-print so output is visible

    with torch.no_grad():
        clean_preds = torch.cat([wrapped_model(images[i:i+BATCH_SIZE]).argmax(1)
                                 for i in range(0, len(images), BATCH_SIZE)])
        adv_preds   = torch.cat([wrapped_model(x_adv[i:i+BATCH_SIZE]).argmax(1)
                                 for i in range(0, len(images), BATCH_SIZE)])

    clean_mask = (clean_preds == labels)
    robust_acc = (adv_preds == labels).float().mean().item()
    asr        = (clean_mask & (adv_preds != labels)).float().sum().item() \
                 / clean_mask.float().sum().item()

    sweep_results[eps_norm] = {
        'robust_acc':  robust_acc,
        'asr':         asr,
        'eps_pixel':   eps_pixel,
        'px_of_255':   px_of_255,
        'log':         log_text,
    }
    print(f'  >>> Robust Acc: {robust_acc*100:.2f}%  |  ASR: {asr*100:.2f}%')

print('\n' + '='*72)
print('Low-ε sweep complete.')


In [ ]:
# Part 2 — Robustness floor identification
print('=' * 60)
print('ROBUSTNESS FLOOR ANALYSIS')
print('=' * 60)

# Smallest eps where ASR >= 99% (fully fragile threshold)
fully_fragile_eps = None
for e in EPS_NORMALIZED:
    if sweep_results[e]['asr'] >= 0.99:
        fully_fragile_eps = e
        break

# Largest eps where ASR < 50% (meaningfully robust threshold)
robust_eps = None
for e in reversed(EPS_NORMALIZED):
    if sweep_results[e]['asr'] < 0.50:
        robust_eps = e
        break

if fully_fragile_eps is not None:
    r = sweep_results[fully_fragile_eps]
    print(f'Smallest eps with ASR >= 99% (fully fragile):  eps={fully_fragile_eps}  '
          f'(pixel={r["eps_pixel"]:.6f}, {r["px_of_255"]:.4f}/255)')
    print(f'  ASR at that eps: {r["asr"]*100:.2f}%')
else:
    print('ASR never reached 99% across the tested range — model shows partial robustness.')

print()

if robust_eps is not None:
    r = sweep_results[robust_eps]
    print(f'Largest eps with ASR < 50%  (meaningfully robust): eps={robust_eps}  '
          f'(pixel={r["eps_pixel"]:.6f}, {r["px_of_255"]:.4f}/255)')
    print(f'  ASR at that eps: {r["asr"]*100:.2f}%')
else:
    print('ASR never dropped below 50% across the tested range.')

print()
min_eps = EPS_NORMALIZED[0]
min_asr = sweep_results[min_eps]['asr']
if min_asr >= 0.99:
    print(f'NOTE: ASR is {min_asr*100:.2f}% at the smallest tested eps ({min_eps}).')
    print('      The robustness floor is BELOW the tested range.')
    print(f'      {MODEL_NAME} is fragile even at sub-pixel perturbations '
          f'({sweep_results[min_eps]["px_of_255"]:.4f}/255).')
else:
    print(f'At eps={min_eps}, ASR = {min_asr*100:.2f}% — floor visible within tested range.')


In [ ]:
# Part 3 — ASR vs eps curve (log-scale x-axis)
eps_vals   = EPS_NORMALIZED
asr_vals   = [sweep_results[e]['asr'] * 100 for e in eps_vals]
pixel_vals = [sweep_results[e]['eps_pixel'] for e in eps_vals]
px255_vals = [sweep_results[e]['px_of_255'] for e in eps_vals]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(eps_vals, asr_vals, 'o-', color='steelblue', linewidth=2, markersize=7, label='ASR')
ax.axhline(99, color='red',    linestyle='--', linewidth=1.2, label='99% ASR (fully fragile)')
ax.axhline(50, color='orange', linestyle='--', linewidth=1.2, label='50% ASR (meaningfully robust)')

ax.set_xlabel('ε (normalized space, log scale)', fontsize=12)
ax.set_ylabel('Attack Success Rate (%)',         fontsize=12)
ax.set_title('AutoAttack Low-ε Sweep — VGG-16 on BelgiumTSC', fontsize=13)
ax.set_ylim(-2, 105)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

# Secondary x-axis labels showing pixel/255 values
ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(eps_vals)
ax2.set_xticklabels([f'{v:.4f}/255' for v in px255_vals], rotation=45, fontsize=7)
ax2.set_xlabel('ε (pixels out of 255)', fontsize=10)

plt.tight_layout()
plt.savefig('vgg16_bel_lowsweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: vgg16_bel_lowsweep.png')


In [ ]:
# Part 4 — Sub-attack cascade breakdown
# Find smallest eps where model is NOT at 100% ASR
breakdown_eps = None
for e in EPS_NORMALIZED:
    if sweep_results[e]['asr'] < 1.0:
        breakdown_eps = e
        break

if breakdown_eps is None:
    print('Model is at 100% ASR for all tested eps values.')
    print('Sub-attack breakdown not applicable — APGD-CE alone suffices down to '
          f'eps={EPS_NORMALIZED[0]}.')
else:
    log = sweep_results[breakdown_eps]['log']
    # Parse lines like: 'robust accuracy after APGD-CE: 13.50% (total time 224.5 s)'
    # and:              'initial accuracy: 97.55%'
    def parse_acc(label, text):
        m = re.search(rf'{re.escape(label)}[:\s]+([\d.]+)%', text, re.IGNORECASE)
        return float(m.group(1)) if m else None

    initial   = parse_acc('initial accuracy', log)
    after_apgd_ce = parse_acc('robust accuracy after APGD-CE', log)
    after_apgd_t  = parse_acc('robust accuracy after APGD-T',  log)
    after_fab     = parse_acc('robust accuracy after FAB-T',   log)
    after_sq      = parse_acc('robust accuracy after SQUARE',  log)
    final         = sweep_results[breakdown_eps]['robust_acc'] * 100

    total_correct = initial  # baseline clean-correct images
    def defeated(before, after):
        if before is None or after is None:
            return 'N/A'
        n = round((before - after) / 100 * len(images))
        return str(n)

    eps_px = sweep_results[breakdown_eps]['eps_pixel']
    px255  = sweep_results[breakdown_eps]['px_of_255']
    print(f'eps={breakdown_eps} (pixel={eps_px:.6f}, {px255:.4f}/255) sub-attack cascade:')
    print(f'  Initial robust acc:   {initial:>8.2f}%')
    if after_apgd_ce is not None:
        print(f'  After APGD-CE:        {after_apgd_ce:>8.2f}%   (defeated ~{defeated(initial, after_apgd_ce)} images)')
    if after_apgd_t is not None:
        print(f'  After APGD-T:         {after_apgd_t:>8.2f}%   (defeated ~{defeated(after_apgd_ce, after_apgd_t)} images)')
    if after_fab is not None:
        print(f'  After FAB-T:          {after_fab:>8.2f}%   (defeated ~{defeated(after_apgd_t, after_fab)} images)')
    if after_sq is not None:
        print(f'  After Square:         {after_sq:>8.2f}%   (defeated ~{defeated(after_fab, after_sq)} images)')
    print(f'  Final robust acc:     {final:>8.2f}%')
    print()
    if after_apgd_t is None and after_apgd_ce is not None and after_apgd_ce <= 0.1:
        print('  => APGD-CE alone achieved near-zero robust acc; targeted/boundary attacks not needed.')
    elif after_apgd_ce is not None and after_apgd_ce > 0.5:
        print('  => APGD-CE was insufficient alone; targeted/boundary attacks (APGD-T/FAB/Square) were needed.')


In [ ]:
# Part 5 — Summary table and JSON export
print('=' * 65)
print(f'  Low-ε AutoAttack Summary')
print('=' * 65)
print(f'  Model:   {MODEL_NAME}')
print(f'  Dataset: {DATASET}')
print()
print(f'  {"eps (norm)":>10} | {"eps (pixel)":>11} | {"px/255":>8} | {"AA ASR":>8} | {"Robust Acc":>10}')
print(f'  {"-"*60}')
for e in EPS_NORMALIZED:
    r = sweep_results[e]
    print(f'  {e:>10.4f} | {r["eps_pixel"]:>11.6f} | {r["px_of_255"]:>8.4f} | '
          f'{r["asr"]*100:>7.2f}% | {r["robust_acc"]*100:>9.2f}%')
print()

# Verdict
min_e  = EPS_NORMALIZED[0]
min_asr = sweep_results[min_e]['asr']
fully_fragile_eps = next((e for e in EPS_NORMALIZED if sweep_results[e]['asr'] >= 0.99), None)
if min_asr >= 0.99:
    verdict = (f'Model remains 100%% fragile down to eps={min_e} '
               f'({sweep_results[min_e]["px_of_255"]:.4f}/255). '
               f'Robustness floor is below the tested range.')
elif fully_fragile_eps is not None:
    verdict = (f'Robustness floor: ASR drops below 99%% below eps={fully_fragile_eps} '
               f'({sweep_results[fully_fragile_eps]["px_of_255"]:.4f}/255).')
else:
    verdict = 'ASR never reached 99%% within the tested range — model retains partial robustness.'
print(f'  Verdict: {verdict}')
print('=' * 65)

# Save JSON for cross-model aggregation
export = {
    'model': MODEL_NAME,
    'dataset': DATASET,
    'sweep': [
        {'eps_norm': e, 'eps_pixel': sweep_results[e]['eps_pixel'],
         'px_of_255': sweep_results[e]['px_of_255'],
         'asr': sweep_results[e]['asr'],
         'robust_acc': sweep_results[e]['robust_acc']}
        for e in EPS_NORMALIZED
    ]
}
with open('lowsweep_vgg16_bel.json', 'w') as f:
    json.dump(export, f, indent=2)
print(f'Results saved to lowsweep_vgg16_bel.json')
